# RAG Pipeline with LangChain and LangSmith
## PDF Document Loading, Chunking, Embedding, Retrieval, and Generation

This notebook covers a complete Retrieval-Augmented Generation (RAG) pipeline using:
- PDF document loaders (PyPDFLoader and PyMuPDFLoader)
- LangChain text splitters for chunking
- OpenAI embeddings
- ChromaDB vector store
- RAG chain with prompt templates
- LangSmith tracing and observability

All code uses the latest LangChain v0.3+ APIs and is tested for VS Code compatibility.

---
## Section 0: Installation and Setup

In [ ]:
# Install all required packages
# Run this cell first before anything else

import subprocess
import sys

packages = [
    "langchain",
    "langchain-openai",
    "langchain-community",
    "langchain-core",
    "langchain-text-splitters",
    "langsmith",
    "chromadb",
    "tiktoken",
    "pypdf",
    "pymupdf",
    "python-dotenv",
]

print("Installing required packages...")
for pkg in packages:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-U", pkg],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    print(f"  Installed: {pkg}")

print("\nAll packages installed successfully.")

In [ ]:
# Environment variable configuration
# Replace placeholder values with your actual API keys before running

import os
from dotenv import load_dotenv

# Load from .env file if it exists (will not overwrite already-set variables)
load_dotenv(override=False)

# OpenAI API key (required for embeddings and LLM)
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY", "sk-...")

# LangSmith configuration (required for tracing)
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] = os.getenv("LANGSMITH_API_KEY", "lsv2_...")
os.environ["LANGSMITH_PROJECT"] = "rag-pipeline-langchain"

# Verify keys are loaded
api_key = os.environ.get("OPENAI_API_KEY", "")
ls_key = os.environ.get("LANGSMITH_API_KEY", "")

if api_key and not api_key.startswith("sk-..."):
    print(f"OpenAI API key loaded: sk-...{api_key[-4:]}")
else:
    print("WARNING: OpenAI API key not set. Update the value above or create a .env file.")

if ls_key and not ls_key.startswith("lsv2_..."):
    print(f"LangSmith API key loaded: lsv2_...{ls_key[-4:]}")
    print(f"LangSmith project: {os.environ['LANGSMITH_PROJECT']}")
    print("LangSmith tracing is ENABLED. All LangChain calls will be traced.")
else:
    print("WARNING: LangSmith API key not set. Tracing will not work.")

---
## Section 1: PDF Document Loading

We demonstrate two PDF loaders:
1. **PyPDFLoader** - Uses pypdf library, good general-purpose loader
2. **PyMuPDFLoader** - Uses PyMuPDF (fitz), faster and better with complex layouts

Both return a list of Document objects, one per page, with page content and metadata.

In [ ]:
# ---------------------------------------------------------------
# PDF Loading with PyPDFLoader
# This is the most commonly used PDF loader in LangChain.
# It extracts text page by page and stores page numbers in metadata.
# ---------------------------------------------------------------

from langchain_community.document_loaders import PyPDFLoader

# Update this path to point to your PDF file
PDF_PATH = "sample.pdf"  # <-- replace with your PDF file path

try:
    loader = PyPDFLoader(PDF_PATH)
    pdf_documents = loader.load()

    print(f"Loaded {len(pdf_documents)} pages from: {PDF_PATH}")
    print(f"\nPage 1 metadata: {pdf_documents[0].metadata}")
    print(f"Page 1 content (first 500 chars):\n{pdf_documents[0].page_content[:500]}")
except FileNotFoundError:
    print(f"File not found: {PDF_PATH}")
    print("Please update PDF_PATH to point to a valid PDF file.")
    print("\nUsing fallback sample text for the rest of this notebook...")

    # Fallback: create a sample document for demonstration
    from langchain_core.documents import Document
    pdf_documents = [
        Document(
            page_content=(
                "LangChain is a framework designed to simplify the creation of applications "
                "using large language models. It provides tools for document loading, text splitting, "
                "embedding generation, vector storage, and retrieval-augmented generation. "
                "RAG (Retrieval-Augmented Generation) is a technique that enhances LLM responses "
                "by first retrieving relevant context from a knowledge base and then generating "
                "answers grounded in that context. This approach reduces hallucinations and allows "
                "the model to answer questions about domain-specific documents it was not trained on. "
                "Key components of a RAG pipeline include: document loaders to ingest data, "
                "text splitters to create manageable chunks, embedding models to convert text to vectors, "
                "vector stores to enable similarity search, and a generation step that combines "
                "retrieved context with the user query to produce accurate answers."
            ),
            metadata={"source": "fallback", "page": 0}
        )
    ]

In [ ]:
# ---------------------------------------------------------------
# PDF Loading with PyMuPDFLoader (alternative, faster loader)
# PyMuPDFLoader uses the PyMuPDF (fitz) library under the hood.
# It is generally faster and handles complex PDF layouts better.
# ---------------------------------------------------------------

from langchain_community.document_loaders import PyMuPDFLoader

try:
    mu_loader = PyMuPDFLoader(PDF_PATH)
    mu_documents = mu_loader.load()

    print(f"PyMuPDFLoader: Loaded {len(mu_documents)} pages")
    print(f"\nPage 1 metadata keys: {list(mu_documents[0].metadata.keys())}")
    print(f"Page 1 content (first 300 chars):\n{mu_documents[0].page_content[:300]}")
except FileNotFoundError:
    print("Skipping PyMuPDFLoader demo (no PDF file found).")
    print("Using PyPDFLoader results (or fallback) for the rest of this notebook.")

In [ ]:
# ---------------------------------------------------------------
# Loading multiple PDFs from a directory
# Use PyPDFDirectoryLoader when you have a folder of PDFs.
# ---------------------------------------------------------------

from langchain_community.document_loaders import PyPDFDirectoryLoader

# Update this path to a directory containing PDF files
PDF_DIRECTORY = "./pdfs/"  # <-- replace with your PDF directory path

try:
    dir_loader = PyPDFDirectoryLoader(PDF_DIRECTORY)
    dir_documents = dir_loader.load()
    print(f"Loaded {len(dir_documents)} pages from directory: {PDF_DIRECTORY}")
    for doc in dir_documents[:3]:
        print(f"  Source: {doc.metadata.get('source', 'unknown')}, Page: {doc.metadata.get('page', 'N/A')}")
except Exception as e:
    print(f"Directory loading skipped: {e}")
    print("This is expected if the directory does not exist.")

---
## Section 2: Text Splitting (Chunking)

Large documents must be split into smaller chunks for effective retrieval.
We use RecursiveCharacterTextSplitter which tries to split on natural boundaries
(paragraphs, sentences, words) while respecting chunk size limits.

In [ ]:
# ---------------------------------------------------------------
# Text Splitting with RecursiveCharacterTextSplitter
# This splitter tries to keep semantically related text together
# by splitting on paragraph breaks first, then sentences, then words.
# ---------------------------------------------------------------

from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # Maximum characters per chunk
    chunk_overlap=100,    # Overlap between consecutive chunks for context continuity
    length_function=len,
    separators=["\n\n", "\n", ". ", " ", ""],  # Priority order of split points
)

# Split the loaded PDF documents into chunks
chunks = text_splitter.split_documents(pdf_documents)

print(f"Original document pages: {len(pdf_documents)}")
print(f"Chunks after splitting: {len(chunks)}")
print(f"\nChunk 1 ({len(chunks[0].page_content)} chars):")
print(chunks[0].page_content[:300])
print(f"\nChunk 1 metadata: {chunks[0].metadata}")

---
## Section 3: Embeddings

We use OpenAI's embedding model to convert text chunks into numerical vectors.
These vectors capture semantic meaning and enable similarity-based retrieval.

In [ ]:
# ---------------------------------------------------------------
# Initialize OpenAI Embeddings
# The text-embedding-3-small model provides a good balance of
# quality and cost for most RAG applications.
# ---------------------------------------------------------------

from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small"
)

# Test the embedding model with a sample text
sample_text = "What is retrieval-augmented generation?"
sample_embedding = embedding_model.embed_query(sample_text)

print(f"Embedding model: text-embedding-3-small")
print(f"Embedding dimension: {len(sample_embedding)}")
print(f"Sample embedding (first 5 values): {sample_embedding[:5]}")

---
## Section 4: Vector Store with ChromaDB

ChromaDB is an open-source vector database that stores embeddings and supports
fast similarity search. We create a collection from our document chunks.

In [ ]:
# ---------------------------------------------------------------
# Create ChromaDB Vector Store from Document Chunks
# This embeds all chunks and stores them for retrieval.
# ---------------------------------------------------------------

from langchain_community.vectorstores import Chroma

# Create vector store from the document chunks
# This will embed each chunk using the embedding model
chroma_store = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    collection_name="rag_pdf_collection",
)

print(f"Vector store created with {chroma_store._collection.count()} documents")

In [ ]:
# ---------------------------------------------------------------
# Create a Retriever from the Vector Store
# The retriever will return the top-k most similar chunks
# for a given query.
# ---------------------------------------------------------------

retriever = chroma_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3},  # Return top 3 most relevant chunks
)

# Test the retriever with a sample query
test_query = "What is RAG?"
retrieved_docs = retriever.invoke(test_query)

print(f"Query: {test_query}")
print(f"Retrieved {len(retrieved_docs)} documents:\n")
for i, doc in enumerate(retrieved_docs):
    print(f"--- Result {i+1} (Source: {doc.metadata.get('source', 'N/A')}) ---")
    print(doc.page_content[:200])
    print()

---
## Section 5: RAG Chain with LangChain

We build the complete RAG chain:
1. User asks a question
2. Retriever finds relevant chunks
3. Chunks are formatted as context
4. LLM generates an answer grounded in the context

All calls are automatically traced by LangSmith.

In [ ]:
# ---------------------------------------------------------------
# Define the RAG Prompt Template
# The prompt instructs the LLM to answer based only on the
# provided context, reducing hallucinations.
# ---------------------------------------------------------------

from langchain_core.prompts import ChatPromptTemplate

rag_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant that answers questions based on the provided context. "
     "If the context does not contain enough information to answer the question, "
     "say so clearly. Do not make up information."),
    ("human",
     "Context:\n{context}\n\n"
     "Question: {question}\n\n"
     "Answer:"),
])

print("RAG prompt template created.")
print(f"Input variables: {rag_prompt.input_variables}")

In [ ]:
# ---------------------------------------------------------------
# Initialize the LLM (Chat Model)
# Using GPT-4o-mini for cost-effective generation.
# ---------------------------------------------------------------

from langchain_openai import ChatOpenAI

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.0,  # Deterministic output for reproducibility
)

print(f"LLM initialized: {llm.model_name}")

In [ ]:
# ---------------------------------------------------------------
# Build the RAG Chain using LCEL (LangChain Expression Language)
# This chain: retrieves context -> formats prompt -> calls LLM -> parses output
# ---------------------------------------------------------------

from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


def format_docs(docs):
    """Concatenate retrieved document contents into a single context string."""
    return "\n\n".join(doc.page_content for doc in docs)


# Build the chain using the pipe operator
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("RAG chain built successfully.")
print("Chain components: Retriever -> Format Docs -> Prompt -> LLM -> Output Parser")

In [ ]:
# ---------------------------------------------------------------
# Run RAG Queries
# Each query is automatically traced in LangSmith.
# ---------------------------------------------------------------

def ask_rag(question):
    """Run a question through the RAG chain and print the result."""
    print(f"Question: {question}")
    print("-" * 60)
    answer = rag_chain.invoke(question)
    print(f"Answer: {answer}")
    print("=" * 60)
    print()
    return answer


# Test queries
answer1 = ask_rag("What is LangChain?")
answer2 = ask_rag("How does RAG work?")
answer3 = ask_rag("What are the key components of a RAG pipeline?")

---
## Section 6: LangSmith Tracing and Observability

LangSmith automatically traces all LangChain calls when LANGSMITH_TRACING=true is set.
Below we demonstrate explicit tracing with the @traceable decorator for custom functions.

In [ ]:
# ---------------------------------------------------------------
# LangSmith Client Setup and Verification
# ---------------------------------------------------------------

from langsmith import Client as LangSmithClient

ls_client = LangSmithClient()

# Verify connection to LangSmith
try:
    # List recent runs to confirm API key works
    projects = list(ls_client.list_projects(limit=1))
    print("LangSmith connection verified successfully.")
    print(f"Project: {os.environ.get('LANGSMITH_PROJECT', 'default')}")
    print("All RAG chain calls above have been traced.")
    print("View traces at: https://smith.langchain.com")
except Exception as e:
    print(f"LangSmith connection issue: {e}")
    print("Tracing may not be working. Check your LANGSMITH_API_KEY.")

In [ ]:
# ---------------------------------------------------------------
# Explicit Tracing with @traceable Decorator
# Use this to trace custom (non-LangChain) functions in LangSmith.
# ---------------------------------------------------------------

from langsmith import traceable


@traceable(name="CustomRAGQuery", run_type="chain")
def traced_rag_query(question):
    """Run a RAG query with explicit LangSmith tracing.
    This creates a named span in LangSmith for easier debugging."""
    
    # Retrieve relevant documents
    docs = retriever.invoke(question)
    context = format_docs(docs)
    
    # Generate answer
    answer = rag_chain.invoke(question)
    
    return {
        "question": question,
        "context_chunks": len(docs),
        "answer": answer,
    }


# Run a traced query
result = traced_rag_query("What is the purpose of text splitting in RAG?")

print("Traced RAG Query Result:")
print(f"  Question: {result['question']}")
print(f"  Context chunks used: {result['context_chunks']}")
print(f"  Answer: {result['answer']}")
print()
print("This trace is visible in LangSmith under the 'CustomRAGQuery' span.")

In [ ]:
# ---------------------------------------------------------------
# Selective Tracing with tracing_context
# You can enable or disable tracing for specific code blocks.
# ---------------------------------------------------------------

import langsmith as ls

# This block will be traced even if LANGSMITH_TRACING is not globally set
with ls.tracing_context(enabled=True, project_name="rag-pipeline-langchain"):
    traced_answer = rag_chain.invoke("Summarize the main topics in this document.")
    print(f"Traced answer: {traced_answer[:200]}...")

print()
print("Selective tracing block completed.")
print("Check LangSmith for the trace under project: rag-pipeline-langchain")

---
## Summary

This notebook demonstrated a complete RAG pipeline with LangChain and LangSmith:

**Document Loading**: PyPDFLoader, PyMuPDFLoader, and PyPDFDirectoryLoader for PDF ingestion.

**Text Splitting**: RecursiveCharacterTextSplitter for creating overlapping chunks.

**Embeddings**: OpenAI text-embedding-3-small for semantic vector representations.

**Vector Store**: ChromaDB for storing and searching embeddings.

**RAG Chain**: LCEL-based chain combining retriever, prompt template, LLM, and output parser.

**Observability**: LangSmith automatic tracing, @traceable decorator, and selective tracing.